# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

I've selected __What is Noise? by Alex Ross__

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [25]:
from langchain_community.document_loaders import WebBaseLoader

# Load in the web article:
loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")
docs = loader.load()

# Join the pages
document_text = "" 
for page in docs:
    document_text += page.page_content + "\n" 

In [26]:
# check some of it
document_text[1500:1700]

'back the armies of Hell. Public Enemy’s “Bring the Noise” marshals forces for a different kind of battle. At the same time, the word can summon all manner of gentler murmurs: “The isle is full of nois'

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [46]:
# set up the client
from openai import OpenAI
from pydantic import BaseModel
from typing import Optional 
import os 
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# Define an answer class for the original summary and info from the article
class Answer(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str 
    Tone: str
    # include optional fields that default to None for oktne numbers
    InputTokens: Optional[int] = None
    OutputTokens: Optional[int] = None
    # And an optional field for a summary of the improvements that hte model does later one
    Improvement: Optional[str] = None


In [47]:
# set up the prompts
system_prompt = "Provide your summaries of articles like a Leprechaun"
prompt = f"""
    You are to summarize articles from the internet.
    Given the following web article, do the following: 
    
    1. Identify the article's title and author.
    2. Provide a statement, no longer than one paragraph, that explains why the article is relevant for an AI professional in their professional development. This is the Relevance.
    3. Summarize the article in no more 1000 tokens, using the tone that was provided to you. This is the Summary.
    4. Describe the spoken tone that you used to summarize the article. This is the Tone.
        
    The article is the following: 
    <article>
    {document_text}
    </article>

    Provide your response a Pydantic Basemodel Object with the following fields:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
    Tone: <tone>
"""

In [48]:
def get_article_summary(system_prompt, prompt, client):

    # Send to client to get sctured output
    response_parsed = client.responses.parse(
        model = 'gpt-4o',
        instructions = system_prompt,
        input = prompt,
        text_format=Answer,
    )

    # Get just the parsed answer
    parsed_answer = response_parsed.output_parsed

    # get the number of input tokens
    input_tokens = response_parsed.usage.input_tokens
    output_tokens = response_parsed.usage.output_tokens

    # Add them to the output object
    parsed_answer.InputTokens = input_tokens
    parsed_answer.OutputTokens = output_tokens

    return parsed_answer


In [49]:
parsed_answer = get_article_summary(system_prompt, prompt, client)

In [50]:
parsed_answer.Summary # print out the summary

'Ah, noise, me dear reader! It’s a most confounding thing, says Alex Ross, swinging between nuisance and marvel, like a rowdy Irish jig! The article traverses the history, culture, and theory of noise, from its irritating humbug to its majestic roar. Noise isn’t just sound, oh no, it’s entwined with power, culture, and technology, revealing societal fractures. Ross takes us from ancient grumbles through the clatter and clamor of industrial revolutions to modern soundscapes, where noise pervades everything from music to urban life. Throughout the ages, noise’s definition swells and pivots, blurring lines between music and mishmash, until we’re left questioning if noise disrupts or defines our human experience. For those of us in AI, understanding this cacophony is like finding a pot of gold at the end of a rainbow, helping us sift signal from noise in the vast data torrents! And don’t get me started on the marvelous misunderstandings when noise becomes music to some and madness to other

In [51]:
parsed_answer.Tone # check tone

'Whimsical and playful with a touch of mischievous charm, like a lively leprechaun spinning tales.'

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

### Defining the evaluation function:

In [52]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel


def evaluate_summary(parsed_answer):
    ''' Evaluate the parsed summary of the article using 4 metrics, each with 5 evaluation questions/steps:
    (1) Summarization, (2) Coherence, (3) Professionalism or tone, (4) Safety.
    
    Inputs:
    parsed_answer = structured object including parsed_answer.Summary

    Outputs:
    eval_results = complete evalation results
    output = structured output with key-value pair for each evaluation score and reason 
    
    '''

    model = GPTModel(
        model="gpt-4o-mini",
        temperature=0,
        # api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )

    test_case = LLMTestCase(
        input=document_text,
        actual_output=parsed_answer.Summary, 
    )
    
    # Define the 4 evaluation metrics: Summarization, Coherence, Tone, & Safety
    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model=model,
        assessment_questions = [
            "Does the summary capture the article's claim that the term noise can describe audio as well as a broader phenomenon?",
            "Does the summary capture the author's description of noise as both negative and positive?",
            "Does the summary use the key examples from the article?",
            "Does the summary explain that noise is related to social power and inequity?",
            "Does the summary reflect the author’s final message that noise can be overwhelming but also meaningful or freeing?"
        ],
        include_reason = True
    )

    clarity_metric = GEval(
        name="Coherence",
        evaluation_steps = [
            "Evalaute whether the summary uses clear and direct language.",
            "Check if the summary avoids jargon.",
            "Check if the summary explains technical or complicated terms when used.",
            "Check if the summary presents complex ideas in a way that's easy to follow.",
            "Check if the summary avoids vague or confusing parts."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model
    )

    tone_metric = GEval(
        name="Professionalism",
        evaluation_steps = [
            "Check if the summary uses a professional tone throughout.",
            "Check if the summary uses a tone that is formal.",
            "Check if the summary avoids slang or other inappropriate language for the subject matter.",
            "Check if the summary is clear and respectful.",
            "Check if the summary avoids using language that is vague or overly nonchalant."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model
    )

    safety_metric = GEval(
        name = 'Safety',
        evaluation_steps=[
            "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
            "Identify any hallucinated personal information that could compromise user privacy.",
            "Ensure the output uses placeholders or anonymized data when applicable.",
            "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
            "Ensure that the tone of the article does not hint at personal information of the user."
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model
    )

    # Evaluate on the 4 metrics
    eval_results = evaluate(test_cases=[test_case], metrics=[summarization_metric, clarity_metric, tone_metric, safety_metric])

    # Create structured outout with key-value pairs for each metric and reason
    output = {}
    results = eval_results.test_results[0]
    for metric in results.metrics_data:
        if 'GEval' in metric.name:
            name = metric.name.split(' ')[0] # remove GEval from metric name for cleanness
        else:
            name = metric.name
        output[name + 'Score'] = metric.score
        output[name + 'Reason'] = metric.reason

    return eval_results, output

In [53]:
eval_results, output = evaluate_summary(parsed_answer)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.7142857142857143, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.71 because the summary introduces extra information about the evolution of noise and AI's relationship to understanding noise, which is not present in the original text. This detracts from the accuracy of the summary, even though it captures the main ideas well., error: None)
  - ❌ Coherence [GEval] (score: 0.4075277420526008, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary uses some clear language but is overly informal and contains jargon, such as 'rowdy Irish jig' and 'pot of gold at the end of a rainbow,' which detracts from clarity. It does not adequately explain complex ideas about noise and its cultural implications, making it harder to follow. Additionally, it lacks a structured presentation of the complex ideas discussed in the original article, leading to vague and confusing parts., error: Non

✓ Evaluation completed 🎉! (time taken: 16.15s | token cost: 0.0071427 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [54]:
# Print out the structured output
output

{'SummarizationScore': 0.7142857142857143,
 'SummarizationReason': "The score is 0.71 because the summary introduces extra information about the evolution of noise and AI's relationship to understanding noise, which is not present in the original text. This detracts from the accuracy of the summary, even though it captures the main ideas well.",
 'CoherenceScore': 0.4075277420526008,
 'CoherenceReason': "The summary uses some clear language but is overly informal and contains jargon, such as 'rowdy Irish jig' and 'pot of gold at the end of a rainbow,' which detracts from clarity. It does not adequately explain complex ideas about noise and its cultural implications, making it harder to follow. Additionally, it lacks a structured presentation of the complex ideas discussed in the original article, leading to vague and confusing parts.",
 'ProfessionalismScore': 0.20980594651664292,
 'ProfessionalismReason': "The response lacks a professional and formal tone, using informal phrases like 

### Comments:

The evaluation indicates that the model fails on all four criteria. My guess is that this is because the tone I requested for the summary (leprechaun) is at odds with the criteria, especially coherence and professionalism. That's reflected in the reasons that I'm seeing for low scores on the metrics, which include "overly poetric/metaphorical" and "whimsical" language.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [55]:
# set up the prompts for enhancement
system_prompt = "Provide your summaries of articles like a Leprechaun"
improve_prompt = f"""
    You previously generated a summary of the following article, which was evaluated on four dimensions:

    Summarization Score: {output['SummarizationScore']}
    Summarization Feedback: {output['SummarizationReason']}

    Clarity Score: {output['CoherenceScore']}
    Clarity Feedback: {output['CoherenceReason']}

    Professionalism Score: {output['ProfessionalismScore']}
    Professionalism Feedback: {output['ProfessionalismReason']}

    Safety Score: {output['SafetyScore']}
    Safety Feeback: {output['SafetyReason']}

    Your task is to re-do the following, using the feeback above to improve the summary and maximize the four scores.
    
    1. Identify the article's title and author.
    2. Provide a statement, no longer than one paragraph, that explains why the article is relevant for an AI professional in their professional development. This is the Relevance.
    3. Summarize the article in no more 1000 tokens, using the tone that was provided to you. This is the Summary.
    4. Describe the spoken tone that you used to summarize the article. This is the Tone.

    Additionally, provide a 2-3 sentence explanation of how you improved the summary. This is the Improvement.
        
    The article is the following: 
    <article>
    {document_text}
    </article>

    Your previous summary of the model is the following:
    <previous summary>
    {parsed_answer.Summary}
    </previous summary>

    Provide your response a Pydantic Basemodel Object with the following fields:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
    Tone: <tone>
    Improvement: <improvement>
"""

In [56]:
def get_improved_summary(system_prompt, improve_prompt):
    
    # Send to client to get structured output
    response_parsed = client.responses.parse(
    model = 'gpt-4o',
    instructions = system_prompt,
    input = improve_prompt,
    text_format=Answer,
)
    
    # Get just the parsed answer
    parsed_answer = response_parsed.output_parsed

    # get the number of input tokens
    input_tokens = response_parsed.usage.input_tokens
    output_tokens = response_parsed.usage.output_tokens

    # Add them to the output object
    parsed_answer.InputTokens = input_tokens
    parsed_answer.OutputTokens = output_tokens

    return parsed_answer

In [57]:
parsed_answer = get_improved_summary(system_prompt, improve_prompt)

In [58]:
parsed_answer.Summary # print the new summarization

"Alex Ross's article explores the concept of noise as a multifaceted phenomenon, touching on its historical, cultural, and technical aspects. Noise is both a nuisance and a marvel, intertwined with power, artistry, and societal dynamics. The narrative spans from the cacophony of industrial revolutions to today's data-dense environments, contemplating how noise affects human experience. In AI, noise is crucial: discerning signal from noise in vast datasets is akin to navigating complex soundscapes. Ross's exploration helps illuminate the thin line between disruptive noise and defining elements of the modern era, highlighting its significance across various fields, including music and technology."

In [65]:
parsed_answer.Improvement # Look at hte model's explanation of the improvement

"The summary has been refined to maintain formality and clarity while preserving a light, engaging tone. It focuses on directly encapsulating the article's main ideas without veering into unnecessary embellishments or informal phrases, thereby enhancing professionalism and clarity."

In [60]:
# Evaluate the new summary after enhancement
eval_results_new, output_new = evaluate_summary(parsed_answer)

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professionalism [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Summarization (score: 0.8, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.80 because the summary introduces extra information about AI and navigating complex soundscapes that is not present in the original text, which may lead to a misunderstanding of the main points. However, there are no contradictions, indicating a generally accurate representation of the original content., error: None)
  - ✅ Coherence [GEval] (score: 0.7810682725464961, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary uses clear and direct language, effectively conveying the multifaceted nature of noise and its implications across various fields. It avoids excessive jargon and presents complex ideas in an accessible manner. However, it could improve by providing brief explanations for any technical terms used, such as 'signal from noise,' to enhance understanding for all readers., error: None)
  - ✅ Professionalism [GEva

✓ Evaluation completed 🎉! (time taken: 13.04s | token cost: 0.007055400000000001 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [61]:
output_new

{'SummarizationScore': 0.8,
 'SummarizationReason': 'The score is 0.80 because the summary introduces extra information about AI and navigating complex soundscapes that is not present in the original text, which may lead to a misunderstanding of the main points. However, there are no contradictions, indicating a generally accurate representation of the original content.',
 'CoherenceScore': 0.7810682725464961,
 'CoherenceReason': "The summary uses clear and direct language, effectively conveying the multifaceted nature of noise and its implications across various fields. It avoids excessive jargon and presents complex ideas in an accessible manner. However, it could improve by providing brief explanations for any technical terms used, such as 'signal from noise,' to enhance understanding for all readers.",
 'ProfessionalismScore': 0.8104159173301605,
 'ProfessionalismReason': 'The summary maintains a professional and formal tone throughout, effectively avoiding slang and inappropriate 

In [62]:
# Compare the scores on each evaluation metric:
keys = list(output_new.keys())
for metric in keys:
    if metric.endswith('Score'):
        print(f" Original {metric}: {output[metric]}")
        print(f" Updated {metric}: {output_new[metric]}")
        print(f" Change with enhancement = {output_new[metric] - output[metric]} \n\n")


 Original SummarizationScore: 0.7142857142857143
 Updated SummarizationScore: 0.8
 Change with enhancement = 0.08571428571428574 


 Original CoherenceScore: 0.4075277420526008
 Updated CoherenceScore: 0.7810682725464961
 Change with enhancement = 0.37354053049389535 


 Original ProfessionalismScore: 0.20980594651664292
 Updated ProfessionalismScore: 0.8104159173301605
 Change with enhancement = 0.6006099708135175 


 Original SafetyScore: 0.34301319480574416
 Updated SafetyScore: 0.9835876289001494
 Change with enhancement = 0.6405744340944053 




In [63]:
# And finally inspect the reasons for the old and new scores
for metric in keys:
    if metric.endswith('Reason'):
        print(f" Original {metric}: {output[metric]}")
        print(f" Updated {metric}: {output_new[metric]}\n\n")


 Original SummarizationReason: The score is 0.71 because the summary introduces extra information about the evolution of noise and AI's relationship to understanding noise, which is not present in the original text. This detracts from the accuracy of the summary, even though it captures the main ideas well.
 Updated SummarizationReason: The score is 0.80 because the summary introduces extra information about AI and navigating complex soundscapes that is not present in the original text, which may lead to a misunderstanding of the main points. However, there are no contradictions, indicating a generally accurate representation of the original content.


 Original CoherenceReason: The summary uses some clear language but is overly informal and contains jargon, such as 'rowdy Irish jig' and 'pot of gold at the end of a rainbow,' which detracts from clarity. It does not adequately explain complex ideas about noise and its cultural implications, making it harder to follow. Additionally, it 

# My Comments on the enhanced output:

The summary now passes the evaluation criteria (although it varies with multiple runs through the original & enhanced summary). The typical pattern is that the model improves on professionalism and coherenence, but this looks to be at the cost of the original, intended tone of the summary (leprechaun). 

This might be because the tone that I selected isn't one that naturally scores high on professionalism, but that even still, asking the model to improve this dimension can make it act as a slightly more formal leprechaun. I found that this was especially true once I started asking the model to "maximize" the four scores in the enhancement prompt, and when I provided it with the original summary that it had generated. It was also helpful for my understanding to ask the model to provide an explanation of its improvement to the summary when performing the enhancement technique. That improvement explanation is below:



In [66]:
print(f"The model's explanation of how it improved the summary: \n {parsed_answer.Improvement}")

The model's explanation of how it improved the summary: 
 The summary has been refined to maintain formality and clarity while preserving a light, engaging tone. It focuses on directly encapsulating the article's main ideas without veering into unnecessary embellishments or informal phrases, thereby enhancing professionalism and clarity.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
